# Test de Conexión con Claude API

Este notebook prueba la conexión con la API de Claude utilizando la clave API desde un archivo .env.

In [ ]:
# Instalar dependencias necesarias (ejecutar si no están instaladas)
!pip install python-dotenv requests

In [ ]:
import os
import requests
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que la clave API existe
api_key = os.getenv("CLAUDE_API_KEY")
if not api_key:
    raise ValueError("❌ CLAUDE_API_KEY no está definida en el archivo .env")
else:
    # Solo mostrar los primeros 5 caracteres por seguridad
    print(f"✓ CLAUDE_API_KEY encontrada: {api_key[:5]}...{api_key[-2:]}")

In [ ]:
# Función para probar diferentes métodos de autenticación
def test_claude_api(auth_method="x-api-key"):
    base_url = "https://api.anthropic.com/v1"
    model = "claude-3-haiku-20240307"
    prompt = "¿Quién eres tú?"
    
    # Configurar encabezados según el método de autenticación
    if auth_method == "x-api-key":
        headers = {
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    else: # "bearer"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    
    body = {
        "model": model,
        "max_tokens": 300,
        "temperature": 0.7,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        print(f"Intentando método: {auth_method}")
        response = requests.post(f"{base_url}/messages", headers=headers, json=body)
        
        # Mostrar detalles de la respuesta
        print(f"Código de estado: {response.status_code}")
        if response.status_code != 200:
            print(f"Detalles del error: {response.text}")
            return None
        
        data = response.json()
        return data["content"][0]["text"]
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

In [ ]:
# Probar con x-api-key (método oficial según la documentación)
response_xapi = test_claude_api("x-api-key")
if response_xapi:
    print("\n✅ Método x-api-key funcionó correctamente")
    print(f"Respuesta: {response_xapi}")

In [ ]:
# Probar con Authorization: Bearer (método alternativo)
response_bearer = test_claude_api("bearer")
if response_bearer:
    print("\n✅ Método Bearer funcionó correctamente")
    print(f"Respuesta: {response_bearer}")

## Implementación completa una vez que sabemos qué método funciona

Después de determinar qué método de autenticación funciona, puedes usar la siguiente implementación completa:

In [ ]:
def generate_text_with_claude(prompt, model="claude-3-haiku-20240307", max_tokens=300, temperature=0.7):
    # Usar el método que funcionó en las pruebas anteriores
    # Cambia 'x-api-key' por 'bearer' si ese fue el que funcionó
    auth_method = "x-api-key"  # o "bearer"
    
    base_url = "https://api.anthropic.com/v1"
    
    if auth_method == "x-api-key":
        headers = {
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    else: # "bearer"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        }
    
    body = {
        "model": model,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        response = requests.post(f"{base_url}/messages", headers=headers, json=body)
        response.raise_for_status()
        data = response.json()
        return data["content"][0]["text"]
    except Exception as e:
        print(f"❌ Error: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Detalles: {e.response.text}")
        return None

In [ ]:
# Probar la implementación final
respuesta = generate_text_with_claude("Explica brevemente qué es una API REST")
print(respuesta)

## Diagnóstico de problemas comunes

Si sigues teniendo problemas, aquí hay algunas soluciones:

1. **Error 401 (Unauthorized)**: Verifica que tu clave API sea correcta y esté activa.
2. **Error 403 (Forbidden)**: Tu clave API no tiene permisos para acceder al recurso solicitado.
3. **Error 429 (Too Many Requests)**: Has alcanzado el límite de solicitudes. Espera un tiempo antes de intentar de nuevo.
4. **Problemas con el modelo**: Asegúrate de estar utilizando un modelo disponible en tu cuenta.